In [2]:
import numpy as np
import random
import time
from numba import jit, prange
import matplotlib.pyplot as plt
import os
import matplotlib.colors as colors
from matplotlib.gridspec import GridSpec

# 预先生成邻居矩阵的函数无需过度优化，因为它只在初始化时运行一次
@jit(nopython=True)
def init_neighbor(N, L):
    neigh = np.zeros((N, 4), dtype=np.int32)
    di = np.array([1, -1, L, -L], dtype=np.int32)
    for i in range(N):
        for j in range(4):
            it = ((i + di[j]) % N + N) % N
            neigh[i, j] = it
    return neigh

# 核心主函数：开启并行(parallel=True)与激进数学优化(fastmath=True)
@jit(nopython=True, parallel=True, fastmath=True)
def PD_Q_game_optimized(N, epsilon, gamma, alpha, Max_t, L, x, c, c_max):
    # 1. 初始化变量 (严格指定数据类型，避免类型推断开销)
    percent = np.zeros(Max_t, dtype=np.float64)
    Repu_avg = np.zeros(Max_t, dtype=np.float64)
    
    Q_table = np.random.randn(N, 10, 2)
    Action = np.random.randint(0, 2, N).astype(np.int32)
    State = np.zeros(N, dtype=np.int32)
    State_new = np.zeros(N, dtype=np.int32)
    Reward = np.zeros(N, dtype=np.float64)
    Re = np.random.random(N)
    Action_0 = np.copy(Action)
    neig = init_neighbor(N, L)
    
    # 2. 查表法替代函数调用：预计算 2x2 收益矩阵
    # payoff_matrix[A1, A2], 0=D(背叛), 1=C(合作)
    # DD=0, DC=1+x, CD=-x, CC=1
    payoff_matrix = np.empty((2, 2), dtype=np.float64)
    payoff_matrix[0, 0] = 0.0          
    payoff_matrix[0, 1] = 1.0 + x      
    payoff_matrix[1, 0] = -x           
    payoff_matrix[1, 1] = 1.0          
    half = L // 2

    for i in range(N):
        x = i // L
        y = i % L
        if x < half and y < half:
            Q_table[i][:,1] = 0.2
            Q_table[i][:,0] = 0
            Re[i] = 1
        elif x >= half and y < half:
            Q_table[i][:,1] = 0
            Q_table[i][:,0] = 0.2
            Re[i] = 1
        elif x < half and y >= half:
            Q_table[i][:,0] = 0.2
            Q_table[i][:,1] = 0
            Re[i] = 0
        else:
            Q_table[i][:,0] = 0
            Q_table[i][:,1] = 0.2
            Re[i] = 0
    
    # 初始化 State
    for i in range(N):
        itC_num = 0
        for j in range(4):
            itC_num += Action[neig[i, j]]
        State[i] = Action[i] + itC_num

    # 3. 核心时间步循环
    for t in range(Max_t):
        if t == 1:
            Action_1 = np.copy(Action)
        if t == 10000000:
            Action_10000000 = np.copy(Action)
        if t == 30000000:
            Action_30000000 = np.copy(Action)
        # --- 循环1：动作选择与声誉更新 (融合循环，并行) ---
        for i in prange(N):
            # Epsilon-greedy 动作选择 (摒弃 argmax)
            p = np.random.random()
            if p < epsilon:
                a = random.randint(0, 1)
            else:
                # 手工比较，速度极快
                if Q_table[i, State[i], 0] >= Q_table[i, State[i], 1]:
                    a = 0
                else:
                    a = 1
            Action[i] = a
            
            # 声誉更新内联 (Inlining)，摒弃额外函数调用
            if a == 1:
                new_repu = Re[i] + c
            else:
                new_repu = Re[i] - c
                
            # 边界截断
            if new_repu > c_max:
                new_repu = c_max
            elif new_repu < 0.0:
                new_repu = 0.0
            Re[i] = new_repu
            
        # --- 循环2：环境交互，计算收益与新状态 (并行) ---
        for i in prange(N):
            payoff = 0.0
            itC_num = 0
            a1 = Action[i]
            # 循环展开：邻居总是4个
            for j in range(4):
                neighbor_idx = neig[i, j]
                a2 = Action[neighbor_idx]
                itC_num += a2
                payoff += payoff_matrix[a1, a2] # O(1)查表
            
            Reward[i] = payoff * 0.25
            State_new[i] = a1 + itC_num
            
        # --- 循环3：适应度计算与 Q-table 更新 (并行) ---
        for i in prange(N):
            repu_sum = 0.0
            reward_neighbor_sum = 0.0
            for j in range(4):
                neighbor_idx = neig[i, j]
                repu_sum += Re[neighbor_idx]
                reward_neighbor_sum += Reward[neighbor_idx]
                
            avg_repu = repu_sum * 0.25
            avg_reward_neighbor = reward_neighbor_sum * 0.25
            
            # 动态适应度计算
            fitness = (1.0 - avg_repu) * Reward[i] + avg_repu * avg_reward_neighbor
            
            # 获取下一状态的最大 Q 值 (摒弃 max)
            q_0 = Q_table[i, State_new[i], 0]
            q_1 = Q_table[i, State_new[i], 1]
            if q_0 > q_1:
                qmax_new = q_0
            else:
                qmax_new = q_1
                
            # 更新 Q-table
            s_i = State[i]
            a_i = Action[i]
            Q_table[i, s_i, a_i] = (1.0 - alpha) * Q_table[i, s_i, a_i] + \
                                   alpha * (fitness + gamma * qmax_new)
            
            # 状态转移
            State[i] = State_new[i]
            
        # --- 宏观统计量 (利用 Numpy C底层加速) ---
        percent[t] = np.sum(Action) / N
        Repu_avg[t] = np.sum(Re) / N
        
    return percent, Action_0, Action_1, Action_10000000, Action_30000000, Action

In [ ]:
if __name__ == "__main__":
    # 模拟参数
    L = 100          # 网格扩大到 100x100 (10,000个节点)，感受下速度！
    N = L * L
    epsilon = 0.01
    gamma = 0.9
    alpha = 0.1
    Max_t = 50000000     # 迭代 5000 步
    x = 0.5
    c = 0.5
    c_max = 1.0

    print(f"开始模拟: {N} 个智能体, 迭代 {Max_t} 步...")
    
    # 预热 Numba 编译器 (第一次调用会慢，因为要编译)
    print("Numba 编译中...")
    # 正式计时
    start_time = time.time()
    percent, Action_0, Action_1, Action_10000000, Action_30000000, Action = PD_Q_game_optimized(N, epsilon, gamma, alpha, Max_t, L, x, c, c_max)
    end_time = time.time()
    
    print(f"模拟完成！极限优化耗时: {end_time - start_time:.4f} 秒")